### MultiQuery Retriever (MultiQueryRetriever)

Distance-based vector search can fail if a user query prompt is phrased differently from the indexed documents.

MultiQueryRetriever uses an LLM (ChatGoogleGenerativeAI) to automatically generate multiple alternative query variations for a given user prompt.

### How It Works

1. Query Expansion: Passes original user prompt to LLM to generate 3 different perspectives/rephrasings.
2. Parallel Retrieval: Runs vector search for each generated query variation in parallel.
3. Result Union: Takes the unique union of all retrieved documents, removing duplicates.

In [2]:
from dotenv import load_dotenv, find_dotenv
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_core.documents import Document

load_dotenv(find_dotenv())

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

docs = [
    Document(page_content="Vector stores utilize HNSW graph indexing for fast approximate nearest neighbor search."),
    Document(page_content="Dense embedding models convert words into floating point numerical coordinate vectors."),
    Document(page_content="Generative artificial intelligence uses transformer neural networks for text generation.")
]

# Ephemeral in-memory Chroma instance
vectorstore = Chroma.from_documents(docs, embeddings)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

# Initialize MultiQueryRetriever
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

results = multiquery_retriever.invoke("How do vector databases search embeddings?")

print(f"MultiQueryRetriever generated queries and retrieved {len(results)} document(s):")
for i, doc in enumerate(results, 1):
    print(f"Doc {i}: {doc.page_content}")


MultiQueryRetriever generated queries and retrieved 1 document(s):
Doc 1: Vector stores utilize HNSW graph indexing for fast approximate nearest neighbor search.
